## Function 3: Mask NDVI and Build a Time Series 📈

In this notebook, you'll learn how to build the `mask_ndvi_by_field()` and `build_ndvi_time_series()` functions step by step. This is where you combine raster and vector data to summarize NDVI inside polygons and track vegetation change through time.

### 🎯 What This Function Does
- Mask NDVI arrays using polygon geometry
- Extract raster values inside vector features
- Build time-series summaries from multiple scenes
- Calculate mean NDVI through time
- Create NDVI line plots for temporal analysis

### 🔧 Function Signature
```python
def mask_ndvi_by_field(image, field_geom):
    """
    Args:
        image (dict): Scene dictionary containing NDVI and transform
        field_geom: Polygon geometry used for masking
    
    Returns:
        numpy.ma.MaskedArray: NDVI values masked to the polygon
    """
```

```python
def build_ndvi_time_series(images, fields):
    """
    Args:
        images (list): Scene dictionaries
        fields (geopandas.GeoDataFrame): Polygon features
    
    Returns:
        dict: NDVI values through time by field
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/rasterio_basics.py`  
**Function Names**: `mask_ndvi_by_field()`, `build_ndvi_time_series()`  
**Replace**: The placeholder functions with your working code

---

### 📚 Step 1: Load Polygon Data and Scene Data

Let's load the polygon features we will use to summarize raster values.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import geopandas as gpd
import rasterio
from rasterio.plot import show
from rasterio import features
from pyproj import Transformer
from pystac_client import Client
import planetary_computer
import matplotlib.dates as mdates
from datetime import datetime

stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
bbox = [-110.74772571639987, 32.270431012618026, -110.70996215904789, 32.29386169894274]
date_range = "2020-01-01/2021-12-31"

catalog = Client.open(stac_url)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=date_range,
    query={"eo:cloud_cover": {"lt": 5}}
)
items = list(search.items())

def windowed_read(url, bbox):
    signed_url = planetary_computer.sign(url)
    with rasterio.open(signed_url) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        left, bottom = transformer.transform(bbox[0], bbox[1])
        right, top = transformer.transform(bbox[2], bbox[3])
        row_start, col_start = src.index(left, top)
        row_stop, col_stop = src.index(right, bottom)
        window = rasterio.windows.Window.from_slices(
            (row_start, row_stop),
            (col_start, col_stop)
        )
        pixels = src.read(1, window=window)
        transform = src.window_transform(window)
        return pixels, transform

images = []
for item in items:
    red_url = item.assets["B04"].href
    nir_url = item.assets["B08"].href
    visual_url = item.assets["visual"].href
    
    red_pixels, transform = windowed_read(red_url, bbox)
    nir_pixels, _ = windowed_read(nir_url, bbox)
    
    ndvi = (nir_pixels.astype(float) - red_pixels.astype(float)) / (
        nir_pixels.astype(float) + red_pixels.astype(float)
    )
    
    signed_visual_url = planetary_computer.sign(visual_url)
    with rasterio.open(signed_visual_url) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        left, bottom = transformer.transform(bbox[0], bbox[1])
        right, top = transformer.transform(bbox[2], bbox[3])
        row_start, col_start = src.index(left, top)
        row_stop, col_stop = src.index(right, bottom)
        window = rasterio.windows.Window.from_slices(
            (row_start, row_stop),
            (col_start, col_stop)
        )
        rgb = src.read([1, 2, 3], window=window)
    
    images.append({
        "date": item.datetime.strftime("%Y-%m-%d"),
        "rgb": rgb,
        "ndvi": ndvi,
        "transform_window": transform
    })

fields = gpd.read_file('neighborhood_samples.geojson').to_crs(epsg=32612)

print(f"✅ Loaded {len(images)} scenes")
print(f"✅ Loaded {len(fields)} polygons")

### 🗺️ Step 2: Visualize Polygons on Raster Imagery

Let's see where the polygons fall on top of the imagery.

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 10))
show(images[0]['rgb'], transform=images[0]['transform_window'], ax=ax, alpha=0.75)
fields.boundary.plot(ax=ax, color='skyblue', linewidth=3)
ax.set_title("Polygons over Imagery")
plt.tight_layout()
plt.show()

### ✂️ Step 3: Build a Raster Mask

**💡 This will be used in our function!**

In [ ]:
field_geom = fields.iloc[0].geometry

mask = features.geometry_mask(
    [field_geom],
    out_shape=(images[0]['ndvi'].shape[0], images[0]['ndvi'].shape[1]),
    transform=images[0]['transform_window'],
    all_touched=False,
    invert=False
)

print(f"✅ Mask shape: {mask.shape}")
print(f"✅ Mask dtype: {mask.dtype}")

### 🌿 Step 4: Mask NDVI Inside One Polygon

Let's test masking with one image and one field first.

In [ ]:
masked_ndvi = ma.masked_array(images[0]['ndvi'], mask)

fig, ax = plt.subplots(figsize=(10, 10))
show(images[0]['rgb'], transform=images[0]['transform_window'], ax=ax, alpha=0.5)
show(masked_ndvi, transform=images[0]['transform_window'], ax=ax, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_title(f"Masked NDVI ({images[0]['date']})")
plt.tight_layout()
plt.show()

print(f"✅ Mean NDVI inside polygon: {masked_ndvi.mean():.3f}")

### 🔁 Step 5: Apply the Mask Across All Fields and Images

Now let's repeat that workflow for all scenes and all polygons.

In [ ]:
for image in images:
    ndvi_fields = {}
    for index, row in fields.iterrows():
        mask = features.geometry_mask(
            [row.geometry],
            out_shape=(image['ndvi'].shape[0], image['ndvi'].shape[1]),
            transform=image['transform_window'],
            all_touched=False,
            invert=False
        )
        ndvi_fields[row.area] = ma.masked_array(image['ndvi'], mask)
    image['ndvi_fields'] = ndvi_fields

print(f"✅ Added masked NDVI for {len(fields)} fields across {len(images)} images")

### 📊 Step 6: Calculate Mean NDVI Through Time

**💡 This will be used in our function!**

In [ ]:
time_series_data = {}

for field_id in fields['area']:
    dates = []
    ndvi_values = []
    
    for image in images:
        dates.append(image['date'])
        ndvi_values.append(image['ndvi_fields'][field_id].mean())
    
    time_series_data[field_id] = {
        'dates': dates,
        'ndvi': ndvi_values
    }

print(f"✅ Built time series for {len(time_series_data)} fields")
print(f"✅ Example field keys: {list(time_series_data.keys())}")

### 📈 Step 7: Plot the NDVI Time Series

Let's create a line graph to show how NDVI changes through time.

In [ ]:
for field_id in time_series_data:
    time_series_data[field_id]['dates_dt'] = [
        datetime.strptime(d, "%Y-%m-%d") for d in time_series_data[field_id]['dates']
    ]

fig, ax = plt.subplots(figsize=(14, 8))

for field_id, data in time_series_data.items():
    ax.plot(
        data['dates_dt'],
        data['ndvi'],
        marker='o',
        linewidth=2,
        markersize=4,
        label=field_id
    )

ax.set_xlabel('Date')
ax.set_ylabel('Mean NDVI')
ax.set_title('NDVI Time Series by Field', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 🧩 Step 8: Building the Complete Functions

Now let's put everything together into reusable functions. This is what you will implement in `src/rasterio_basics.py`.

In [ ]:
import numpy.ma as ma
from rasterio import features
from datetime import datetime

def mask_ndvi_by_field(image, field_geom):
    """
    Mask NDVI values to a polygon.
    """
    mask = features.geometry_mask(
        [field_geom],
        out_shape=(image['ndvi'].shape[0], image['ndvi'].shape[1]),
        transform=image['transform_window'],
        all_touched=False,
        invert=False
    )
    return ma.masked_array(image['ndvi'], mask)

def build_ndvi_time_series(images, fields):
    """
    Build mean NDVI through time for each field.
    """
    for image in images:
        ndvi_fields = {}
        for index, row in fields.iterrows():
            ndvi_fields[row.area] = mask_ndvi_by_field(image, row.geometry)
        image['ndvi_fields'] = ndvi_fields
    
    time_series_data = {}
    for field_id in fields['area']:
        dates = []
        ndvi_values = []
        for image in images:
            dates.append(image['date'])
            ndvi_values.append(image['ndvi_fields'][field_id].mean())
        time_series_data[field_id] = {
            'dates': dates,
            'ndvi': ndvi_values,
            'dates_dt': [datetime.strptime(d, "%Y-%m-%d") for d in dates]
        }
    
    return time_series_data

### ✨ Step 9: Test Your Functions

Let's test our complete functions with raster scenes and polygon features.

In [ ]:
# Test 1: Mask one field
print("Test 1: Mask one field")
masked = mask_ndvi_by_field(images[0], fields.iloc[0].geometry)
print(f"✅ Masked array type: {type(masked)}")
print(f"✅ Mean NDVI: {masked.mean():.3f}")
print()

# Test 2: Build time series
print("Test 2: Build time series")
ts_data = build_ndvi_time_series(images, fields)
print(f"✅ Fields in time series: {list(ts_data.keys())}")
example_key = list(ts_data.keys())[0]
print(f"✅ Dates for first field: {len(ts_data[example_key]['dates'])}")
print(f"✅ NDVI values for first field: {len(ts_data[example_key]['ndvi'])}")
print()

print("🎉 All tests completed!")

### 🧪 Step 10: Verify with Pytest

Test your entire functions to verify your implementation in `src/rasterio_basics.py` by running the following line in the terminal

```bash
uv run pytest tests/ -k "TestMaskNDVIByField or TestBuildNDVITimeSeries" -v
```

**⚠️ IMPORTANT: Make sure this passes before you move on!**

---

### 🔑 Key Learning Points

- **Raster masks** allow us to isolate pixels inside polygon features
- **`geometry_mask()`** is used to connect vector geometry and raster grids
- Masked arrays allow raster statistics to ignore pixels outside the study area
- **Mean NDVI** can be calculated for each polygon across many scenes
- Time-series plots help reveal seasonal vegetation patterns and change over time